# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fasiuu/FLYRANK_TASK1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup — shared feature builder from this repo + DuckDB connection to the hosted release.
# Token: Colab Secret HF_TOKEN (preferred) or a getpass prompt. Never paste a token into a cell.
%pip -q install duckdb
import os, sys, subprocess, json, warnings
warnings.filterwarnings('ignore')
if not os.path.exists('FLYRANK_TASK1'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/Fasiuu/FLYRANK_TASK1.git'], check=True)
sys.path.insert(0, 'FLYRANK_TASK1/work/scripts')
import numpy as np, pandas as pd
import flyrank_features as ff
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
np.random.seed(ff.SEED)
con = ff.connect()
os.makedirs('work/outputs', exist_ok=True)
print('connected; seed =', ff.SEED)

Hugging Face READ token (hf_...): ··········
connected; seed = 42


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector comes from the shared builder in `work/scripts/flyrank_features.py` (one DuckDB scan, aggregated per page). Engineering choices:

- impressions and clicks are `log1p`-ed (heavy tails),
- momentum is a log-ratio between 30-day blocks, so a page going 10→20 and one going 10k→20k look the same,
- position is impression-weighted and only uses days with impressions,
- `spike_share` = biggest single day ÷ the 30-day total, `weekend_share` = weekend impressions ÷ total,
- metadata gaps get a `has_` flag rather than a silent zero.

No categorical one-hot is needed: `content_type` only drives the missingness flags.

In [2]:
# Build the per-item panel for the two development decision dates (one DuckDB scan, ~5-8 min).
import time
t0 = time.time()
panel = ff.build_panel(con, ff.DEV_DATES)
panel['decision_date'] = pd.to_datetime(panel['decision_date']).dt.strftime('%Y-%m-%d')
m = ff.model_frame(panel)
print(f'{len(panel):,} item-date rows built in {time.time()-t0:.0f}s; {len(m):,} pass the population checks')
X = m[ff.FEATURES]
print(X.shape)
print(X.describe(percentiles=[.5, .9]).T.round(3).to_string())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

207,726 item-date rows built in 135s; 134,757 pass the population checks
(134757, 18)
                         count        mean         std         min       50%        90%         max
log_imp_last30        134757.0    6.732975    1.523526    0.693147  6.605298   8.806724   13.591565
log_imp_prev30        134757.0    6.761071    1.550489         0.0  6.693324   8.764053   13.326479
log_imp_prev60_90     134757.0    6.330896    1.940249         0.0   6.43935   8.618196   12.669932
imp_momentum          134757.0   -0.028096    0.892509    -6.49224 -0.042127   0.782759    8.969796
imp_momentum_long     134757.0    0.402079    1.379854   -7.080334  0.201637   1.728722   11.233939
log_clicks_last30     134757.0    1.032996    1.222893         0.0  0.693147   2.890372    8.913954
ctr_last30            134757.0    0.210273    0.386837         0.0  0.078431   0.566572   33.333333
pos_last30            134757.0   15.981758   15.330901         0.0  9.607477  37.286891  106.890995
pos_change    

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

For each feature: what it means, how gaps are handled, and when it exists. All of them are computed from days on or before T, or from metadata that doesn't change with traffic.

`pos_change` and `pos_volatility` are filled with 0 when they can't be measured: no position in the previous month, or only one visible day. In both cases "no measurable change" is the honest reading.

In [3]:
notes = pd.DataFrame({'feature': ff.FEATURES,
                      'missing_in_panel': [round(float(panel[f].isna().mean()), 3) if f in panel else None for f in ff.FEATURES]})
notes['available_at_T'] = 'yes - days <= T, or static metadata'
notes['fill'] = 'none needed'
notes.loc[notes.feature.isin(['pos_change', 'pos_volatility']), 'fill'] = '0 when not measurable'
notes.loc[notes.feature.isin(['log_word_count', 'log_search_volume']), 'fill'] = '0 + has_ flag'
print(notes.to_string(index=False))

             feature  missing_in_panel                      available_at_T                  fill
      log_imp_last30             0.000 yes - days <= T, or static metadata           none needed
      log_imp_prev30             0.000 yes - days <= T, or static metadata           none needed
   log_imp_prev60_90             0.000 yes - days <= T, or static metadata           none needed
        imp_momentum             0.000 yes - days <= T, or static metadata           none needed
   imp_momentum_long             0.000 yes - days <= T, or static metadata           none needed
   log_clicks_last30             0.000 yes - days <= T, or static metadata           none needed
          ctr_last30             0.000 yes - days <= T, or static metadata           none needed
          pos_last30             0.013 yes - days <= T, or static metadata           none needed
          pos_change             0.107 yes - days <= T, or static metadata 0 when not measurable
      pos_volatility          

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I attack my own feature set three ways, all scored with the same client-grouped, time-forward split as the model:

1. **Harness test.** Adding the outcome month's impressions should push AUC towards 1. If it didn't, my test setup would be broken. It does jump.
2. **`updated_after_T`.** Pages edited after T are less likely to fall behind (people edit pages they care about). That is information from after the decision date, so it is excluded on principle. Adding it doesn't help the model here, which is reassuring but doesn't change the rule.
3. **Label siblings.** No feature is built from `imp_next30`, `page_ratio` or `client_ratio`.

**A leak I found in my own earlier work:** in notebook 03, my `pos_volatility` was a standard deviation over the *whole* 180 days, including the 90-day outcome window, and the query-table columns overlapped June. So the 86.5% / 84.0% accuracies I quoted in w01 were measured with leaky features. The numbers from w05 onward replace them.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
MAKERS = {
    'logreg': lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    'hgb': lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                                  min_samples_leaf=100, random_state=ff.SEED),
    'rf': lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=50, n_jobs=-1, random_state=ff.SEED),
}
FINAL = 'hgb'    # frozen in w05 on the development split (best top-of-queue precision), never re-chosen later
F = ff.FEATURES
from sklearn.metrics import roc_auc_score
T1, T2 = ff.DEV_DATES
def auc_with(df, cols):
    o = ff.grouped_forward_scores(df, [T1], T2, MAKERS['hgb'], cols)
    return round(float(roc_auc_score(o.declined.astype(int), o.score)), 4)
d = m.copy(); d['LEAK_log_imp_next30'] = np.log1p(d.imp_next30)        # deliberately leaky: the outcome month itself
res = {'honest feature set': auc_with(m, F),
       '+ outcome-month impressions (harness test)': auc_with(d, F + ['LEAK_log_imp_next30']),
       '+ updated_after_T (export-time edit date)': auc_with(m, F + ['updated_after_T'])}
for k, v in res.items():
    print(f'{k:45} ROC-AUC {v}')
print('\ndecline rate by updated_after_T:'); print(m.groupby('updated_after_T').declined.agg(['mean', 'size']).round(3).to_string())

honest feature set                            ROC-AUC 0.7336
+ outcome-month impressions (harness test)    ROC-AUC 0.9118
+ updated_after_T (export-time edit date)     ROC-AUC 0.7301

decline rate by updated_after_T:
                  mean    size
updated_after_T               
0                0.546   32275
1                0.471  102482


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Everything I refused to use, one line each. The short version: anything built from the outcome month, anything stamped at export time, anything from the query table, all IDs, and every product decision field.

In [5]:
excluded = {
  'imp_next30 / page_ratio / client_ratio': 'built from the outcome month - label sources',
  'content_updated_date, last_optimized_date, updated_after_T': 'export-time values; an edit after T is future information',
  'fact_content_query_90d columns': 'fixed 2026-04-02 -> 2026-06-30 window overlaps every label month',
  'trend_direction / trend_pct style fields': 'label-derived by construction',
  'client_hash_id, content_hash_id, url/keyword hashes': 'pseudonyms - grouping and splitting only',
  'ga4_* / sessions_ai / scroll_events': 'GA4 history too short and AI sessions too sparse for these windows',
  'health_score, priority_score, action_type, refresh flags': 'product decisions - not in the release, and would be circular',
}
for k, v in excluded.items():
    print(f'- {k}: {v}')

- imp_next30 / page_ratio / client_ratio: built from the outcome month - label sources
- content_updated_date, last_optimized_date, updated_after_T: export-time values; an edit after T is future information
- fact_content_query_90d columns: fixed 2026-04-02 -> 2026-06-30 window overlaps every label month
- trend_direction / trend_pct style fields: label-derived by construction
- client_hash_id, content_hash_id, url/keyword hashes: pseudonyms - grouping and splitting only
- ga4_* / sessions_ai / scroll_events: GA4 history too short and AI sessions too sparse for these windows
- health_score, priority_score, action_type, refresh flags: product decisions - not in the release, and would be circular


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.